# Ligand-target binding data processing

This notebook processes molecular binding data from multiple sources (ChEMBL, BindingDB, and PDB+PDBBind) to create a unified, high-quality dataset for ligand-target binding data.

---
## 1. Imports

In [ ]:
import re
import math
import pickle
import sqlite3
from pathlib import Path
from itertools import islice
from collections import defaultdict

import pandas as pd
import numpy as np
from tqdm import tqdm
from tqdm.contrib.concurrent import process_map
from Bio.PDB import PDBParser

from rdkit import Chem, RDLogger
from rdkit.Chem import rdmolops, AllChem, MACCSkeys, inchi, rdReducedGraphs
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.DataStructs.cDataStructs import ExplicitBitVect
RDLogger.DisableLog('rdApp.*')


---
## 2. Define constants and parameters

We remove common crystallography artifacts, buffers, and small ions that are not true ligands but appear in crystal structures due to experimental conditions (based on [10.1038/s44386-025-00006-5](https://doi.org/10.1038/s44386-025-00006-5) and [10.1107/S1744309112044387](https://doi.org/10.1107/S1744309112044387)).

In [ ]:
# Measurement type priority (lower number = higher priority)
MEASURE_PRIORITY = {'Ki': 1, 'Kd': 1, 'IC50': 2, 'EC50': 2}

# Database priority for conflict resolution
DB_PRIORITY = {'CHEMBL': 3, 'PDB': 2, 'PDBBind': 1}

# Common crystallography artifacts, solvents, and buffers to exclude
DISCARD_LIST = [
    "1BO", "12P", "13P", "15P", "1PE", "1PG", "1PS", "2PE", "3PG", "7PE", "9PE",
    "ACN", "ACT", "ACY", "BA", "BEN", "BEZ", "BME", "CA", "CCN", "CD",
    "CHAPS", "CHAPSO", "CIT", "CL", "CO", "CO3", "CU", "DIO", "DMS", "DMX",
    "DOD", "DPV", "DPW", "DTT", "EDO", "EOH", "EPE", "ETX", "FE", "FLC",
    "FMT", "GBL", "GOL", "HEPES", "HEZ", "HTO", "IMD", "IPA", "JEF", "K",
    "KH2", "LI", "MES", "MG", "MLA", "MLI", "MN", "MOH", "MPD", "MRD",
    "NA", "NDS", "NH3", "NI", "NO3", "P22", "P33", "P6G", "PB", "PDO",
    "PE1", "PE2", "PE3", "PE4", "PE5", "PE6", "PE7", "PE8", "PE9", "PEG",
    "PG0", "PG4", "PG5", "PG6", "PGE", "PGO", "PGR", "PO4", "POL", "SBT",
    "SIN", "SO3", "SO4", "SR", "TBU", "TLA", "TME", "TRIS", "TRS", "ZN"
]

# Processing parameters
BINDINGDB_CHUNKSIZE = 300000  # Process BindingDB in chunks to manage memory
FINGERPRINT_BATCH_SIZE = 5000  # Batch size for fingerprint generation
MAX_WORKERS = 16  # Parallel processing workers

---
## 3. Utility Functions

These helper functions handle common data processing tasks throughout the pipeline.

In [ ]:
def smiles_to_canon_inchikey(smiles):
    """
    Clean a SMILES string and return both canonical SMILES and InChIKey.
    
    Filters out invalid molecules, keeps largest fragment, and applies
    size/composition filters (10-50 heavy atoms, must contain carbon).
    """
    if pd.isna(smiles):
        return None, None
    
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None
    
    # Keep only the largest fragment (removes salts, counterions)
    fragments = rdmolops.GetMolFrags(mol, asMols=True)
    mol = max(fragments, key=lambda m: m.GetNumAtoms())
    
    # Size filter: between 10-50 heavy atoms
    if mol.GetNumHeavyAtoms() < 10 or mol.GetNumHeavyAtoms() > 50:
        return None, None
        
    # Discard any metals/metalloids
    organic_atoms = {1, 5, 6, 7, 8, 9, 15, 16, 17, 34, 35, 53}
    if any(atom.GetAtomicNum() not in organic_atoms for atom in mol.GetAtoms()):
        return None, None
    
    try:
        canon_smiles = Chem.MolToSmiles(mol, canonical=True)
        inchi_key = inchi.MolToInchiKey(mol)
        return (canon_smiles, inchi_key) if inchi_key else (None, None)
    except Exception:
        return None, None

def parse_affinity_value(val):
    """
    Parse affinity value from BindingDB, handling comparison symbols.
    BindingDB values can be strings like '<10', '>100', '50', etc.
    """
    if pd.isna(val):
        return None, None
    
    # Already numeric
    if isinstance(val, (int, float)):
        return '=', float(val)
    
    val_str = str(val).strip().replace(',', '')
    
    # Parse with comparison symbols (e.g., '<10', '>=50')
    match = re.match(r'([<>]=?)\s*(\d+\.?\d*)', val_str)
    if match:
        symbol, num_str = match.groups()
        return symbol, float(num_str)
    
    # Try plain number
    try:
        return '=', float(val_str)
    except ValueError:
        return None, None


def list_to_bv(bit_list):
    """Convert binary list to RDKit ExplicitBitVect for fingerprint storage."""
    bv = ExplicitBitVect(len(bit_list))
    bv.SetBitsFromList(np.where(bit_list)[0].tolist())
    return bv


def chunks(iterable, size):
    """Yield successive chunks from an iterable for batch processing."""
    it = iter(iterable)
    while True:
        batch = list(islice(it, size))
        if not batch:
            break
        yield batch


def convert_to_nM(value, units):
    """
    Convert affinity value to nanomolar (nM) units.
    """
    units = units.lower()
    conversion_factors = {
        "fm": 1e-6,
        "pm": 1e-3,
        "nm": 1,
        "um": 1e3,
        "µm": 1e3,
        "mm": 1e6
    }
    if units not in conversion_factors:
        raise ValueError(f"Unrecognized unit: {units}")
    return value * conversion_factors[units]


def get_pdb_chains(pdb_file):
    """Extract chain IDs from a PDB file."""
    parser = PDBParser(QUIET=True)
    structure = parser.get_structure('struct', pdb_file)
    chains = {chain.id for model in structure for chain in model}
    return sorted(chains)

---
## 4. Setup file paths and data sources

### Input files

| File | Description | Source |
|------|-------------|---------|
| `chembl_36.db` | ChEMBL SQLite database (30 GB) | [ChEMBL FTP server](https://ftp.ebi.ac.uk/pub/databases/chembl/ChEMBLdb/latest/) |
| `BindingDB_All.tsv` | BindingDB tsv (9 GB) | [BindingDB downloads](https://www.bindingdb.org/rwd/bind/chemsearch/marvin/Download.jsp) |
| `INDEX_general_PL.2020R1.lst` | PDBBind index file (1.5 MB)  | [PDBBind downloads](https://www.pdbbind-plus.org.cn/download) |
| `P-L/` | PDBBind folder with processed structures (16 GB)  | [PDBBind downloads](https://www.pdbbind-plus.org.cn/download) |
| `cc-to-pdb.tsv` | PDB mapping between ligands and PDB structures (3 MB) | [RCSB script](https://github.com/rcsb/rcsb-training-resources/blob/master/example-use-cases/pdb-ligand-composition/generate_pdb_ligand_mappings.py) |
| `pdb_chain_uniprot.csv` | SIFTS mapping between PDB chains and UniProt (33 MB)  | [SIFTS FTP server](https://ftp.ebi.ac.uk/pub/databases/msd/sifts/csv/) |
| `Components-smiles-stereo-oe.smi` | WWPDB Chemical Component Dictionary OpenEye stereo SMILES (7 MB)  | [WWPDB CCD](https://www.wwpdb.org/data/ccd) |

In [ ]:
# Configure input and output directories
ligands_path = Path('raw/ligand_data')
output_path = Path('processed')
output_path.mkdir(parents=True, exist_ok=True)

CHEMBL_DB = ligands_path / 'chembl_36.db'
BINDINGDB_FILE = ligands_path / 'BindingDB_All.tsv'
PDB_RELATIONS_FILE = ligands_path / 'cc-to-pdb.tsv'
PDBBIND_INDEX_FILE = ligands_path / 'INDEX_general_PL.2020R1.lst'
PDBBIND_STRUCTURES_DIR = ligands_path / 'P-L'
PDB_CHAIN_UNIPROT_CSV = ligands_path / 'pdb_chain_uniprot.csv'
PDB_SMILES_FILE = ligands_path / 'Components-smiles-stereo-oe.smi'

---
## 5. Process ChEMBL database

### Query strategy
- **Target scope**
  - Single protein and protein complex targets only
  - High-confidence target assignments (confidence score 9 for single proteins, 7 for complexes)

- **Assay selection**
  - Binding assays only (`assay_type = 'B'`)
  - Exact activity measurements (`standard_relation = '='`)
  - Activities reported in nanomolar (nM) units

- **Activity data quality**
  - Standard potency/affinity metrics only (Ki, Kd, IC50, EC50)
  - Required pChEMBL values
  - Removal of duplicate and non-validated records
  - Exclusion of ambiguous or negative annotations (e.g. inactive, inconclusive, undetermined)

- **Target annotation**
  - ChEMBL target identifiers and preferred target names
  - UniProt accessions for protein identification (when available)
  - Protein classification information

### Processing steps

1. Query SQLite database
2. Validate and canonicalize molecular structures
3. Handle duplicates measurements by prioritizing measurement type (Ki/Kd over IC50/EC50) and lowest affinity value. We only keep one interaction per ligand-target pair.

In [ ]:
print("\n" + "="*70)
print("PROCESSING CHEMBL DATABASE")
print("="*70)
print(f"Loading data from: {CHEMBL_DB}")

# SQL query for high-quality binding data
query = """
SELECT 
    md.chembl_id AS mol_chembl_id,
    cs.canonical_smiles AS smiles,
    cs.standard_inchi_key AS inchi_key,
    td.chembl_id AS target_chembl_id,
    td.pref_name AS target_name,
    td.target_type,
    act.standard_type AS activity_type,
    act.standard_value AS activity_value,
    act.pchembl_value,
    cc.protein_class_id,
    comp_seq.accession AS uniprot_accession
FROM activities act
INNER JOIN molecule_dictionary md ON act.molregno = md.molregno
INNER JOIN compound_structures cs ON act.molregno = cs.molregno
INNER JOIN assays assay ON act.assay_id = assay.assay_id
INNER JOIN target_dictionary td ON assay.tid = td.tid
LEFT JOIN target_components tc ON td.tid = tc.tid
LEFT JOIN component_class cc ON tc.component_id = cc.component_id
LEFT JOIN component_sequences comp_seq ON tc.component_id = comp_seq.component_id
WHERE 
    cs.canonical_smiles IS NOT NULL
    AND td.target_type IN ('SINGLE PROTEIN', 'PROTEIN COMPLEX')
    AND ((td.target_type = 'SINGLE PROTEIN' AND assay.confidence_score = 9)
         OR (td.target_type = 'PROTEIN COMPLEX' AND assay.confidence_score = 7))
    AND assay.assay_type = 'B'
    AND act.standard_units = 'nM'
    AND act.standard_relation = '='
    AND act.standard_type IN ('IC50', 'Ki', 'Kd', 'EC50')
    AND act.pchembl_value IS NOT NULL
    AND (act.data_validity_comment IS NULL OR LOWER(act.data_validity_comment) = 'manually validated')
    AND act.potential_duplicate = 0
    AND (act.activity_comment IS NULL 
         OR (LOWER(act.activity_comment) NOT LIKE '%undetermined%'
             AND LOWER(act.activity_comment) NOT LIKE '%inconclusive%'
             AND LOWER(act.activity_comment) NOT LIKE '%unspecified%'
             AND LOWER(act.activity_comment) NOT LIKE '%inactive%'
             AND LOWER(act.activity_comment) NOT LIKE '%not active%'));
"""

# Execute query
with sqlite3.connect(CHEMBL_DB) as conn:
    df = pd.read_sql_query(query, conn)

print(f"Loaded {len(df):,} raw activity records")

# Clean and canonicalize SMILES
print("Cleaning SMILES and generating InChI keys...")
results = process_map(smiles_to_canon_inchikey,
                      df['smiles'],
                      max_workers=MAX_WORKERS,
                      chunksize=1000,
                      total=len(df),
                      desc="Processing SMILES"
                      )
df['canon_smiles'], df['inchi_key'] = zip(*results)
df = df[df['canon_smiles'].notna()].copy()
df['database'] = 'CHEMBL'
df['measure_priority'] = df['activity_type'].map(MEASURE_PRIORITY)
print(f"After cleaning: {len(df):,} records")

# Deduplicate (keep best measurement per compound-target pair)
print("Deduplicating compound-target pairs...")
df = (df
      .sort_values(by=['mol_chembl_id',
                       'uniprot_accession',
                       'measure_priority',
                       'activity_value'
                       ],
                   ascending=[True, True, True, True]
                   )
      .drop_duplicates(subset=['mol_chembl_id', 'uniprot_accession'], keep='first')
      .drop(columns=['measure_priority'])
      .rename(columns={'mol_chembl_id': 'ligand_id', 'uniprot_accession': 'target_id'})
      )

# Select final columns
chembl_df = df[['ligand_id',
                'canon_smiles',
                'inchi_key',
                'target_id',
                'activity_type',
                'activity_value',
                'database'
                ]].copy()
chembl_df.to_pickle(output_path / 'chembl_db.pkl')

print(f"Final: {len(df):,} records")
print(f"    • Unique compounds: {df['ligand_id'].nunique():,}")
print(f"    • Unique targets: {df['target_id'].nunique():,}")
chembl_df

---
## 6. Process BindingDB

### Processing steps

1. Read TSV file in chunks to manage memory
2. Extract and prioritize affinity measurements
3. Parse comparison operators and numeric values
4. Validate molecular structures and standardize representation
5. Deduplicate entries

In [ ]:
print("\n" + "="*70)
print("PROCESSING BINDINGDB")
print("="*70)

print(f"Loading data from: {BINDINGDB_FILE}")
print(f"(Processing in chunks of {BINDINGDB_CHUNKSIZE:,} rows)")

# Define columns to load
bindingdb_cols =  ['Curation/DataSource',
                   'PubChem CID',  
                   'ChEMBL ID of Ligand',
                   'Ligand SMILES',
                   'Ligand InChI Key',
                   'UniProt (SwissProt) Primary ID of Target Chain 1',
                   'PDB ID(s) of Target Chain 1',
                   'Ki (nM)',
                   'IC50 (nM)',
                   'Kd (nM)',
                   'EC50 (nM)',
                   ]

# Load dataframe in chunks
bindingdb_chunks = []
chunk_iterator = pd.read_csv(BINDINGDB_FILE,
                             sep='\t',
                             usecols=bindingdb_cols,
                             chunksize=BINDINGDB_CHUNKSIZE,
                             low_memory=False
                             )
for chunk in tqdm(chunk_iterator, desc="Reading chunks"):
    bindingdb_chunks.append(chunk)
df = pd.concat(bindingdb_chunks, ignore_index=True)
print(f"Loaded {len(df):,} total records")

# Keep only records with compound and target IDs
df = df.dropna(subset=['PubChem CID', 'UniProt (SwissProt) Primary ID of Target Chain 1']).copy()

# Create ligand ID (prefer ChEMBL ID if available, otherwise use PubChem CID)
df['PubChem CID'] = df['PubChem CID'].apply(lambda x: f"PUBCHEM{int(float(x))}" if pd.notna(x) else None)
df['ligand_id'] = df['ChEMBL ID of Ligand'].combine_first(df['PubChem CID'])

# Standardize column names
df = df.rename(columns={'Curation/DataSource': 'database',
                        'UniProt (SwissProt) Primary ID of Target Chain 1': 'target_id'
                        }
               )
df['database'] = df['database'].apply(lambda x: f"BindingDB ({x})")

# Reshape affinity columns from wide to long format
print("\nExtracting affinity measurements...")
affinity_cols = ['Ki (nM)', 'Kd (nM)', 'IC50 (nM)', 'EC50 (nM)']
df = df.melt(id_vars=df.columns.difference(affinity_cols),
             value_vars=affinity_cols,
             var_name='activity_type',
             value_name='activity_raw'
             )
df = df.dropna(subset=['activity_raw'])

# Parse affinity values
parsed = df['activity_raw'].apply(parse_affinity_value)
df[['relation', 'activity_value']] = pd.DataFrame(parsed.tolist(), index=df.index)

# Keep only exact measurements (filter out <, >, etc.)
df = df[df['relation'] == '='].copy()

# Clean activity type name and add priority
df['activity_type'] = df['activity_type'].str.replace(' (nM)', '', regex=False)
df['measure_priority'] = df['activity_type'].map(MEASURE_PRIORITY)
df = df.dropna(subset=['activity_value', 'measure_priority'])
print(f"{len(df):,} records with valid affinity measurements")

# Clean SMILES
print("\nCleaning SMILES...")
results = process_map(smiles_to_canon_inchikey,
                      df['Ligand SMILES'],
                      max_workers=MAX_WORKERS,
                      chunksize=1000,
                      total=len(df),
                      desc="    Processing SMILES"
                      )

df['canon_smiles'], df['inchi_key'] = zip(*results)
df = df[df['canon_smiles'].notna()].copy()
print(f"After cleaning SMILES: {len(df):,} records")

print("Deduplicating compound-target pairs...")
# Keep best measurement per compound-target pair
df = (df
      .sort_values(by=['ligand_id',
                       'target_id',
                       'measure_priority',
                       'activity_value'
                       ]
                   )
      .groupby(['ligand_id',
                'target_id'
                ],
               as_index=False
               )
      .first()
      .drop(columns=['measure_priority'])
    )


# Select final columns
bindingdb_df = df[['ligand_id',
                   'canon_smiles',
                   'inchi_key',
                   'target_id',
                   'activity_type',
                   'activity_value',
                   'database']].copy()
bindingdb_df.to_pickle(output_path / 'bindingdb_db.pkl')

print(f"Final: {len(bindingdb_df):,} records")
print(f"    • Unique compounds: {bindingdb_df['ligand_id'].nunique():,}")
print(f"    • Unique targets: {bindingdb_df['target_id'].nunique():,}")
bindingdb_df


---
## 7. Process PDB and PDBBind data

### Processing steps

1. Read relations file that maps each cc to a list of PDB structures.
2. Map PDB structures chains to UniProt.
3. Parse PDBBind data.
4. Validate SMILES.
5. Put it all together. Some entries won't have PDBBind affinities.
6. Deduplicate.

In [ ]:
print("\n" + "="*70)
print("PROCESSING PDB LIGAND DATA")
print("="*70)

# Load PDB ligand-protein relationships
print("Loading PDB ligand-protein relationships...")

relations_dict = {}
with open(PDB_RELATIONS_FILE) as f:
    for line in f:
        chem_id, pdb_ids = line.strip().split('\t')
        pdb_ids = pdb_ids.lower().split()
        relations_dict[chem_id] = pdb_ids

print(f"Loaded {len(relations_dict):,} ligand codes")

# Extract binding data from PDBBind
print("\nExtracting binding data from PDBBind...")

# Map PDB IDs to structure directories
pdbbind_dirs = {}
for subdir in PDBBIND_STRUCTURES_DIR.glob('./*'):
    if subdir.is_dir():
        for subsubdir in subdir.glob('./*'):
            if subsubdir.is_dir():
                pdbbind_dirs[subsubdir.stem] = subsubdir

# Load PDB chain to UniProt mapping
pdb_uniprot_df = pd.read_csv(
    PDB_CHAIN_UNIPROT_CSV,
    skiprows=1,
    low_memory=False,
    usecols=['PDB', 'CHAIN', 'SP_PRIMARY']
)
pdb_uniprot_df = pdb_uniprot_df.drop_duplicates(subset=['PDB', 'CHAIN'], keep='first')
pdb_uniprot_dict = (
    pdb_uniprot_df
    .assign(
        PDB=lambda d: d['PDB'].str.lower(),
        CHAIN=lambda d: d['CHAIN'].str.lower(),
        SP_PRIMARY=lambda d: d['SP_PRIMARY'].str.upper()
    )
    .set_index(['PDB', 'CHAIN'])['SP_PRIMARY']
    .to_dict()
)
pdb_uniprot_dict_simple = {}
for pdb_id, group in pdb_uniprot_df.groupby('PDB'):
    uniprot_ids = list(set(group['SP_PRIMARY']))
    if len(uniprot_ids) > 1:
        continue
    pdb_uniprot_dict_simple[pdb_id] = uniprot_ids[0]

# Parse PDBBind affinity data
pdb_bind_dict = defaultdict(dict)

with open(PDBBIND_INDEX_FILE) as f:
    tot_lines = f.readlines()
    for line in tqdm(tot_lines, desc="Parsing PDBBind", total=len(tot_lines)):
        line = line.strip()
        
        # Skip comments and empty lines
        if not line or line.startswith("#"):
            continue
        
        # Split around comment marker
        if "//" not in line:
            continue
        main_part, comment_part = line.split("//", 1)
        main_part = main_part.strip()
        comment_part = comment_part.strip()
        
        # Extract ligand name from comment (in parentheses)
        if "(" not in comment_part or ")" not in comment_part:
            continue
        
        ligand_name = comment_part.split("(", 1)[-1].split(")", 1)[0].strip()
        
        # Skip polymers and complex ligands
        if "-mer" in ligand_name or "-" in ligand_name or "+" in ligand_name:
            continue
        
        # Clean ligand name
        ligand_name_splits = re.split(r'[\/\-\&\+\s]+', ligand_name)
        ligand_clean_names = []
        
        for ligand_split in ligand_name_splits:
            ligand_split = ligand_split.strip().upper()
            if not ligand_split:
                continue
            if ligand_split in ["OX", "SQ", "HQ"] or ligand_split.isdigit():
                continue
            ligand_split = ligand_split.strip('_')
            ligand_clean_names.append(ligand_split)
        
        if not ligand_clean_names:
            continue
        
        # Extract affinity data
        parts = main_part.split()
        if len(parts) < 4:
            continue
        
        affinity_str = parts[3].strip()
        target = parts[0].strip().lower()  # PDB ID
        
        # Parse affinity (e.g., Ki=49uM, Kd<=10nM)
        affinity_pattern = re.compile(r'([A-Za-z0-9]+)(<=|<|=|~|>|>=)([\d.]+)([munfpµM]+)')
        match = affinity_pattern.match(affinity_str)
        
        if not match:
            continue
        
        measurement, operator, value, units = match.groups()
        value = float(value)
        
        if operator != '=':
            continue
        
        # Convert to nM
        try:
            value_nM = convert_to_nM(value, units)
        except ValueError:
            continue
        
        # Get pocket file and extract chains
        pocket_file = pdbbind_dirs.get(target)
        if pocket_file is None:
            continue
        
        pocket_file = pocket_file / f'{target}_pocket.pdb'
        if not pocket_file.is_file():
            continue
        
        uniprot_ids = []
        chains = get_pdb_chains(pocket_file)
        for chain in chains:
            if not chain.strip():
                continue
            uniprot_id = pdb_uniprot_dict.get((target, chain.lower()))
            if uniprot_id:
                uniprot_ids.append(uniprot_id)
        
        # Store for each ligand code
        for ligand_code in ligand_clean_names:
            for uniprot_id in uniprot_ids:
                pdb_bind_dict[ligand_code][(target, uniprot_id)] = (
                    measurement, operator, value_nM, 'nM'
                )

print(f"Extracted binding data for {len(pdb_bind_dict):,} ligands")

# Load PDB ligand SMILES
print("\nLoading PDB ligand SMILES...")

df = pd.read_csv(PDB_SMILES_FILE, sep='\t', header=None, names=['smiles', 'id', 'name'])
df.dropna(inplace=True)

# Keep only ligands in relations_dict
df = df[df['id'].isin(relations_dict)].copy()
print(f"Loaded {len(df):,} ligands with SMILES")

# Clean and standardize structures
print("\nCleaning and standardizing structures...")

# Fix incorrect bond symbols
df['smiles'] = df['smiles'].str.replace("═", "=", regex=False)

# Filter out artifacts and solvents
df = df[~df['id'].isin(DISCARD_LIST)].copy()
print(f"After removing artifacts: {len(df):,} ligands")

# Apply SMILES cleaning
results = process_map(
    smiles_to_canon_inchikey,
    df['smiles'],
    max_workers=MAX_WORKERS,
    chunksize=1000,
    total=len(df),
    desc="  Processing SMILES"
)
df['canon_smiles'], df['inchi_key'] = zip(*results)
df = df[df['canon_smiles'].notna()].copy()
print(f"After cleaning: {len(df):,} ligands")

# Add crystal structures
df['crystals'] = df['id'].map(relations_dict)

# Create standardized records
print("\nCreating standardized records...")
pdb_records = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="  Processing ligands"):
    ligand_id = row['id']
    canon_smiles = row['canon_smiles']
    inchi_key = row['inchi_key']
    crystal_ids = row['crystals']
    
    # Get binding data for this ligand
    binding_data = pdb_bind_dict.get(ligand_id, {})
    
    for crystal_id in crystal_ids:
        binding_keys = [k for k in binding_data.keys() if crystal_id in k]
        
        if not binding_keys:
            
            uniprot_id = pdb_uniprot_dict_simple.get(crystal_id)
            pdb_records.append({
                'ligand_id': f'PDB{ligand_id}',
                'canon_smiles': canon_smiles,
                'inchi_key': inchi_key,
                'target_id': uniprot_id,
                'activity_type': np.nan,
                'activity_value': np.nan,
                'database': 'PDB',
                'pdb_id': crystal_id
            })
            continue
        
        for binding_key in binding_keys:
            crystal_binding = binding_data[binding_key]
            activity_type, relation, activity_value, units = crystal_binding
            
            pdb_records.append({
                'ligand_id': f'PDB{ligand_id}',
                'canon_smiles': canon_smiles,
                'inchi_key': inchi_key,
                'target_id': binding_key[1],
                'activity_type': activity_type,
                'activity_value': activity_value,
                'database': 'PDBBind',
                'pdb_id': crystal_id
            })

# Create DataFrame
pdb_df = pd.DataFrame(pdb_records)

# Deduplicate: keep best measurement per ligand-PDB pair
pdb_df['priority'] = pdb_df['activity_type'].map(MEASURE_PRIORITY)

has_target = pdb_df['target_id'].notna()
no_target  = pdb_df['target_id'].isna()

df_with_target = pdb_df[has_target].copy()
df_with_target = (
    df_with_target
    .sort_values(by=['inchi_key', 'target_id', 'priority', 'activity_value', 'pdb_id'],
                 ascending=[True, True, True, True, True]
                 )
    .groupby(['inchi_key', 'target_id'], as_index=False)
    .first()
)

pdb_df = pd.concat([df_with_target, pdb_df[no_target]], ignore_index=True).drop('priority', axis=1)
pdb_df.to_pickle(output_path / 'pdb_db.pkl')

print(f"\nFinal: {len(pdb_df):,} records")
print(f"    • Unique compounds: {pdb_df['inchi_key'].nunique():,}")
print(f"    • Unique targets: {pdb_df['target_id'].nunique():,}")
pdb_df

---
## 8. Combine all data

### Deduplication Strategy

When the same compound-target pair appears more than once, we use a hierarchical priority system:
1. **PDB structure**: pairs with PDB structure IDs go first (with or without affinity measurements)
2. **Measurement type**: Ki/Kd > IC50/EC50
3. **Database**: PDBBind > PDB > ChEMBL > BindingDB
4. **Affinity value**: Lower values (stronger binding) preferred

In [ ]:

print("\n" + "="*70)
print("COMBINING AND DEDUPLICATING DATA")
print("="*70)

# Combine all datasets
combined_df = pd.concat([chembl_df, bindingdb_df, pdb_df], ignore_index=True, sort=False)
print(f"Total records before deduplication: {len(combined_df):,}")

# Save intermediate file
intermediate_file = ligands_path / 'combined_df_all.pkl'
combined_df.to_pickle(intermediate_file)

# Add priority columns for sorting
combined_df['measure_priority'] = combined_df['activity_type'].map(MEASURE_PRIORITY)
combined_df['db_priority'] = combined_df['database'].map(DB_PRIORITY).fillna(99).astype(int)

# Final deduplication: keep ONE measurement per compound-target-PDB pair
# Priority order: Best measurement type > Lowest activity value > ChEMBL over BindingDB
print("Performing final deduplication...")
combined_df = (
    combined_df
    .sort_values(
        by=['inchi_key', 'target_id', 'pdb_id', 'measure_priority', 'db_priority', 'activity_value'],
        ascending=[True, True, True, True, True, True],
        na_position='last'
    )
    .drop_duplicates(subset=['inchi_key', 'target_id'], keep='first')
    .drop(columns=['measure_priority', 'db_priority'])
)

print(f"\nFinal combined dataset: {len(combined_df):,} records")
print(f"    • Unique compounds: {combined_df['inchi_key'].nunique():,}")
print(f"    • Unique targets: {combined_df['target_id'].nunique():,}")
print(f"    • Unique ligand IDs: {combined_df['ligand_id'].nunique():,}")


# SAVE OUTPUTS
print("\nSaving combined data...")

# Save as Parquet
parquet_file = output_path / 'combined_df.parquet'
combined_df.to_parquet(parquet_file, index=False, compression='snappy')
print(f"Saved Parquet: {parquet_file}")

# Save as CSV
csv_file = output_path / 'combined_df.csv'
combined_df.to_csv(csv_file, index=False)
print(f"Saved CSV: {csv_file}")
combined_df

---
## 9. Generate molecular fingerprints

### Fingerprint types generated

1. **Morgan (ECFP)**: Circular fingerprints based on atom neighborhoods
2. **RDKit**: Topological path-based fingerprints
3. **MACCS**: 166 predefined structural keys
4. **ASP**: Atom-pair shortest path descriptors (via JCompoundMapper)
5. **LSTAR**: Radial atom-pair fingerprints (via JCompoundMapper)
6. **RAD2D**: 2D radial distribution functions (via JCompoundMapper)

In [ ]:
print("\n" + "="*70)
print("GENERATING MOLECULAR FINGERPRINTS")
print("="*70)

# Load the processed data
data_path = output_path / 'combined_df.parquet'
df = pd.read_parquet(data_path)
print(f"Loaded {len(df)} ligand-target records")

# Create output directory for fingerprints
fps_path = Path('db/fps')
fps_path.mkdir(parents=True, exist_ok=True)

# Prepare molecule data (group by InChI key to get unique molecules)
print("\nPreparing molecule data...")
all_mol_data = []
n_mols = df['inchi_key'].nunique()

ligand_name_priority = {'CHEMBL': 3, 'PDB': 1, 'PDBBind': 2}
for inchi_key, group in tqdm(df.groupby('inchi_key'), total=n_mols, desc="Grouping molecules"):

    # Keep one chem ID only
    if len(group) > 1:
        group['db_priority'] = group['database'].map(ligand_name_priority).fillna(99).astype(int)
        group = group.sort_values(by=['db_priority'])

    ligand_id = group['ligand_id'].iloc[0]
    canon_smiles = group['canon_smiles'].iloc[0]

    # Create affinity dictionary for all targets of this molecule
    affinity_dict = {
        (row['target_id'], row['pdb_id'] if pd.notna(row['pdb_id']) else None): 
        (row['activity_type'], '=', row['activity_value'], 'nM', row['database'])
        for _, row in group.iterrows()
    }
    
    all_mol_data.append((ligand_id, canon_smiles, affinity_dict))

print(f"Prepared {len(all_mol_data):,} unique molecules")


print("\n" + "-"*70)
print("FINGERPRINTS")
print("-"*70)

# Define fingerprint types: (name, generator_fn)
pre_fingerprint_types = [
    ("ERG",     lambda mol: rdReducedGraphs.GetErGFingerprint(mol)),
    ("PATTERN", lambda mol: Chem.PatternFingerprint(mol, fpSize=2048)),
]

fingerprint_types = []
for name, gen in pre_fingerprint_types:
    path = fps_path / f'COMBINED_{name}.pkl'
    if path.is_file():
        print(f"Skipping {name}, {path} already exists")
        continue
    else:
        fingerprint_types.append((name, gen))
        

# One list per FP type + scaffold variants
fp_lists = {name: [] for name, _ in fingerprint_types}
fp_lists["SCAFFOLD"] = {name: [] for name, _ in fingerprint_types}

print("\nGenerating fingerprints using RDKit...")
for ligand_id, smiles, affinity_dict in tqdm(all_mol_data, desc="Processing molecules"):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        continue
    try:
        # Bemis-Murcko scaffold
        scaffold = MurckoScaffold.GetScaffoldForMol(mol)
        scaffold_smiles = Chem.MolToSmiles(scaffold) if scaffold.GetNumAtoms() > 0 else None

        for name, gen_fn in fingerprint_types:
            fp = gen_fn(mol)
            fp_lists[name].append((ligand_id, smiles, affinity_dict, fp))

            # Scaffold FP using the corresponding fingerprint type
            scaffold_fp = gen_fn(scaffold) if scaffold_smiles is not None else fp  # if scaffold fp fails, keep full fp
            fp_lists["SCAFFOLD"][name].append((ligand_id, scaffold_smiles, affinity_dict, scaffold_fp))

    except Exception as e:
        print(f"Error processing {ligand_id}: {e}")
        continue

# Save all fingerprints
print("\nSaving RDKit fingerprints...")
for name, _ in fingerprint_types:
    path = fps_path / f'COMBINED_{name}.pkl'
    with open(path, 'wb') as f:
        pickle.dump(fp_lists[name], f)
    print(f"Saved {name}: {path} ({len(fp_lists[name]):,} fingerprints)")

    scaffold_list = fp_lists["SCAFFOLD"][name]
    scaffold_path = fps_path / f'COMBINED_SCAFFOLD_{name}.pkl'
    with open(scaffold_path, 'wb') as f:
        pickle.dump(scaffold_list, f)
    print(f"Saved scaffold ({name}): {scaffold_path} ({len(scaffold_list):,} fingerprints)")


print("\n" + "="*70)
print("PROCESSING COMPLETE")
print("="*70)